# 04b — Feature Engineering (Week 6)

Four cumulative feature sets, each re-trained with all three models (Week 5 hyperparameters):

| Set | Adds | Why it's allowed (exists for any address) |
|---|---|---|
| Base (Week 4) | size, beds, baths, lot, age, coordinates, county / city / ZIP, seasonality | property facts and location |
| + Ratios | bed/bath ratio, lot-to-living-area ratio | arithmetic on property facts |
| + School districts | K-8 and high-school district from a **spatial join** of coordinates against CA School District Areas 2024-25 | district boundaries are fixed by location |
| + Comp PPSF | ZIP median price per sq ft of **prior (training-window) sales**, and the implied comps value for the home | learned on training sales only, cross-fitted so no row sees its own price |

The comp feature is the leakage-safe version of PPSF: it never uses the home's own ClosePrice and is
fit inside the pipeline on training rows only (`CompPPSFEncoder`).

**Excluded on purpose:** the district file also has demographic fields (race/ethnicity shares). Those are
not used. Pricing homes on neighborhood demographic composition raises fair-housing concerns.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idx_avm import config
from idx_avm.pipeline import prepare_dataset
from idx_avm.models import MODEL_GRIDS, fit_and_score, tune_on_validation, feature_importance

X_MONTHS = json.loads((config.PROCESSED_DIR / "split_metadata.json").read_text())["chosen_window_months"]
data = prepare_dataset(week6_features=True)
DF, TEST_MONTH = data.df, data.months[-1]
VAL_MONTH = TEST_MONTH - 1
print(f"{len(DF):,} rows | train window {X_MONTHS} months | validation {VAL_MONTH} | test {TEST_MONTH}")

## 1. School-district spatial join coverage

In [ ]:
for c in ["SchoolDistrictK8", "SchoolDistrictHigh"]:
    print(f"{c}: {DF[c].notna().mean():.1%} of homes matched, {DF[c].nunique()} districts")
med = (DF.groupby("SchoolDistrictHigh")[config.TARGET].agg(sales="size", median="median")
       .query("sales >= 300").sort_values("median"))
pd.concat([med.head(5), med.tail(5)])

## 2. Re-train every model on each feature set

In [ ]:
FEATURE_SETS = {
    "Base (Week 4)": {},
    "+ Ratios": {"ratios": True},
    "+ School districts": {"ratios": True, "school": True},
    "+ Comp PPSF": {"ratios": True, "school": True, "comps": True},
}
best_params = {name: tune_on_validation(DF, TEST_MONTH, X_MONTHS, name, flags={}, log=lambda *_: None)[0] for name in MODEL_GRIDS}
rows, fitted = [], {}
for set_name, flags in FEATURE_SETS.items():
    for name in MODEL_GRIDS:
        val = fit_and_score(DF, VAL_MONTH, X_MONTHS, name, best_params[name], flags)
        test = fit_and_score(DF, TEST_MONTH, X_MONTHS, name, best_params[name], flags)
        fitted[(name, set_name)] = test
        rows.append({"feature_set": set_name, "model": name, "val_MdAPE": val.metrics["MdAPE"],
                     "test_MdAPE": test.metrics["MdAPE"], "test_MAPE": test.metrics["MAPE"],
                     "test_R2": test.metrics["R2"], "test_MAE": test.metrics["MAE"]})
week6 = pd.DataFrame(rows)
week6.pivot(index="feature_set", columns="model", values="test_MdAPE").loc[list(FEATURE_SETS)].round(2)

In [ ]:
week6.pivot(index="feature_set", columns="model", values="test_R2").loc[list(FEATURE_SETS)].round(3)

## 3. Old vs new feature set (deliverable table)

In [ ]:
old = week6[week6.feature_set == "Base (Week 4)"].set_index("model")
new = week6[week6.feature_set == "+ Comp PPSF"].set_index("model")
pd.DataFrame({
    "MdAPE old": old.test_MdAPE, "MdAPE new": new.test_MdAPE, "Δ MdAPE (pts)": new.test_MdAPE - old.test_MdAPE,
    "R² old": old.test_R2, "R² new": new.test_R2, "Δ R²": new.test_R2 - old.test_R2,
}).round(3)

## 4. Selected model (by validation MdAPE, not test)

In [ ]:
best = week6.loc[week6.val_MdAPE.idxmin()]
print("Selected:", best.model, "with", best.feature_set)
imp = feature_importance(fitted[("Random Forest", "+ Comp PPSF")])
imp.plot.barh(x="feature", y="importance", figsize=(7, 5), legend=False).invert_yaxis()
plt.title("Random Forest importance, full Week 6 feature set"); plt.show()

## 5. Takeaways (fill in from the run)

- Which additions helped which model, and by how much.
- Whether the school-district layer adds information beyond ZIP/city (they overlap heavily).
- Whether comps help the tree models or mostly the linear model (trees can already learn ZIP-level price levels).
- Feature set and model carried into Week 7 (gradient boosting).